In [ ]:
import os

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error


def get_data_from_kaggle():
    from kagglehub.datasets import dataset_download
    downloaded_path = dataset_download(
        "blastchar/telco-customer-churn",
        path="WA_Fn-UseC_-Telco-Customer-Churn.csv",
        output_dir=".",
    )
    os.rename(downloaded_path, "./telco.csv")

In [ ]:
# ==================== 0. Check if the dataset exists, if not download it from Kaggle ====================
if not os.path.exists(r"./telco.csv"):
    print("download")
    get_data_from_kaggle()

assert os.path.exists(r"./telco.csv"), "Dataset not found. Please download it from Kaggle."

In [ ]:
# ==================== 1. Load and inspect the data ====================
data = pd.read_csv(r"./telco.csv")
data = data.convert_dtypes()

# print(data.head())
# data.info()

# ---------- 1.1 change `No internet service` to `No`.
data.replace({ "No internet service": "No", "No phone service": "No"}, inplace=True)

# ---------- 1.2 Convert Yes/No and Male/Female to boolean(integer).
for col in data.columns:
    st = set(data[col].unique())
    if st == {"Yes", "No"}: data[col] = data[col].map({"Yes": 1, "No": 0})
    elif st == {"Male", "Female"}: data[col] = data[col].map({"Female": 0, "Male": 1})


data.info()

In [ ]:
# ==================== 2. Define the variables ====================
# ---------- 2.1 One-hot encode the categorical variables.
data = pd.get_dummies(data, columns=["InternetService"], drop_first=True)
bool_cols = data.select_dtypes(include=["boolean"]).columns
data[bool_cols] = data[bool_cols].astype("Int64")

# ---------- 2.2 Define the features and target variable.
service_cols = [
    "PhoneService", "MultipleLines", "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies", "InternetService_Fiber optic", "InternetService_No"
]
features = data[service_cols]
target = data["MonthlyCharges"]

# features.info()
# target.info()

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(features, target, train_size=0.8, random_state=818)


# ==================== 4. Fit a multivariable linear regression ====================
model = LinearRegression()
model.fit(X_train, y_train)


# ==================== 5. Report the parameters ====================
# print("Coefficients:", model.coef_)
# print("Intercept:", model.intercept_)

b = model.intercept_  # base plan charge
coefficient_list = (pd.DataFrame({'service': service_cols, 'cost': model.coef_})
                    .sort_values('cost', ascending=False)
                    .reset_index(drop=True))  # dollar contribution of each service

print(f"Base monthly charge (intercept): ${b:.2f}")
print(coefficient_list)

In [ ]:
# ==================== 6. Evaluate on the test set ====================
mae = mean_absolute_error(y_test, model.predict(X_test))
mse = mean_squared_error(y_test, model.predict(X_test))
rmse = np.sqrt(mse)
r2 = r2_score(y_test, model.predict(X_test))
print(f"Mean Absolute Error: {mae:.2f}")
print(f"Root Mean Squared Error: {rmse:.2f}")
print(f"R-squared: {r2:.2f}")

In [ ]:
# ==================== 7. Compare to a baseline ====================
# ---------- 7.1 Data preparation
addon_cols = ["MultipleLines", "OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"]
data_new = pd.DataFrame({
    "num_addons": data[addon_cols].sum(axis=1),  # sum across the row = how many add-ons
    "MonthlyCharges": data["MonthlyCharges"],
})

# ---------- 7.2 Split the data and train a baseline model
X_train_new, X_test_new, y_train_new, y_test_new = train_test_split(data_new[["num_addons"]], data_new["MonthlyCharges"], train_size=0.8, random_state=818)
base_model = LinearRegression()
base_model.fit(X_train_new, y_train_new)

# ---------- 7.3 Evaluate the baseline model
mae_base = mean_absolute_error(y_test_new, base_model.predict(X_test_new))
mse_base = mean_squared_error(y_test_new, base_model.predict(X_test_new))
rmse_base = np.sqrt(mse_base)
r2_base = r2_score(y_test_new, base_model.predict(X_test_new))
print(f"Baseline Mean Absolute Error: {mae_base:.2f}")
print(f"Baseline Root Mean Squared Error: {rmse_base:.2f}")
print(f"Baseline R-squared: {r2_base:.2f}")

In [ ]:
# ==================== 8. Interpret the results ====================
coef_table = pd.Series(model.coef_, index=service_cols).sort_values(ascending=False)
print(f"Base monthly charge (intercept): ${model.intercept_:.2f}")
print("Estimated $/month contribution of each service:")
for name, coef in coef_table.items(): print(f"  {name}: {coef:+.2f} $/month")

# higher R2 is better. (Multi Variable Model R2 is higher, so it's better)
# lower MAE & RMSE is better. (Multi Variable Model MAE and RMSE is lower, so it's better)

In [ ]:
# ==================== 9. Predict a new customer ====================
new_customer = pd.DataFrame([{
    "PhoneService": 1,
    "MultipleLines": 1,
    "OnlineSecurity": 1,
    "OnlineBackup": 0,
    "DeviceProtection": 0,
    "TechSupport": 1,
    "StreamingTV": 1,
    "StreamingMovies": 0,
    "InternetService_Fiber optic": 1,
    "InternetService_No": 0,
}], columns=service_cols)

predicted_charge = model.predict(new_customer)[0]
print(f"Predicted monthly charge for the new customer: ${predicted_charge:.2f}")